In [ ]:
# ============================
# Imports + basic configuration
# ============================
import json
from pathlib import Path
from typing import List, Set

# Example config (CHANGE THESE)
SEQ_DIR = Path("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/sequences/isi_2/len120")
BATCH_SIZE = 8  # every 8 sequences form a balanced batch

In [ ]:
# ============================
# JSON helper functions
# ============================

def load_json_list(path: Path) -> List[str]:
    """Load a JSON list of strings, or [] if missing."""
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a JSON list")
    return [str(x) for x in data]


def save_json_list(path: Path, data: List[str]) -> None:
    """Save a list of strings to JSON."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

In [ ]:
# ============================
# Discover and sort sequences
# ============================

def discover_sequences_sorted(seq_dir: Path) -> List[Path]:
    """
    Find all *.json sequence files in seq_dir, excluding
    bookkeeping dirs ('used/' and 'unused/') and sort them.
    """
    all_json = list(seq_dir.rglob("*.json"))
    seqs = []
    for p in all_json:
        if "used" in p.parts or "unused" in p.parts:
            continue
        seqs.append(p)
    seqs.sort(key=lambda p: p.name)
    return seqs

# TEST OUTPUT
all_sequences = discover_sequences_sorted(SEQ_DIR)
len(all_sequences), all_sequences[:5]

In [ ]:
# ============================
# Compute current batch
# ============================

def compute_current_batch(
    all_seqs: List[Path],
    used_basenames: Set[str],
    batch_size: int,
) -> List[str]:
    """
    Given sorted sequences + used.json, determine which batch should
    be active now. Batches are sequential chunks of size batch_size.
    """
    basenames = [p.name for p in all_seqs]
    n = len(basenames)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        chunk = basenames[start:end]
        chunk_set = set(chunk)

        # If this chunk is fully used, skip it
        if chunk_set.issubset(used_basenames):
            continue
        
        # Otherwise, return only the unused filenames from this chunk
        return [f for f in chunk if f not in used_basenames]

    return []  # all batches fully used

# TEST with no used sequences
used_list = load_json_list(SEQ_DIR / "used" / "used.json")
used_set = set(used_list)
compute_current_batch(all_sequences, used_set, BATCH_SIZE)[:BATCH_SIZE]

In [ ]:
# ============================
# Write current batch to unused.json
# ============================

def update_unused_json(seq_dir: Path, batch_size: int) -> List[str]:
    """
    Computes current batch and writes it to unused/unused.json.
    Returns the batch list.
    """
    used_json = seq_dir / "used" / "used.json"
    unused_json = seq_dir / "unused" / "unused.json"

    used_list = load_json_list(used_json)
    used_set = set(used_list)

    all_seqs = discover_sequences_sorted(seq_dir)

    batch = compute_current_batch(all_seqs, used_set, batch_size)
    save_json_list(unused_json, batch)
    return batch

# RUN the batch update
batch = update_unused_json(SEQ_DIR, BATCH_SIZE)
batch

In [ ]:
# ============================
# Check consistency
# ============================

print("Total sequences:", len(all_sequences))
print("Used sequences :", len(used_set))
print("Current batch  :", len(batch))
print("Batch contents :", batch)

In [ ]:
# ============================
# OPTIONAL: simulate marking sequences as used
# ============================

def mark_as_used(seq_dir: Path, seqs: List[str]):
    """
    Add sequences to used/used.json (simulates QC-passed participants).
    """
    path = seq_dir / "used" / "used.json"
    used_list = set(load_json_list(path))
    used_list.update(seqs)
    save_json_list(path, sorted(list(used_list)))

# Example: mark first sequence in batch as used
if batch:
    mark_as_used(SEQ_DIR, [batch[0]])
    print("Marked used:", batch[0])

# Refresh batch after marking
batch_after = update_unused_json(SEQ_DIR, BATCH_SIZE)
batch_after

In [ ]:
SEQ_DIR